In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import os
from sklearn.model_selection import train_test_split

# Define dataset paths
TRAIN_DIR = 'C:/DUT/Ki_2/Edge_AI/dataset/train'
TEST_DIR = 'C:/DUT/Ki_2/Edge_AI/dataset/test'

# Parameters
IMG_SIZE = 32
NUM_CLASSES = 10  # Assuming 10 classes based on the problem description
BATCH_SIZE = 32
VAL_SPLIT = 0.3
EPOCHS = 50 # Reduced epochs for faster execution during testing

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VAL_SPLIT,
    subset="training",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    color_mode='grayscale'
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VAL_SPLIT,
    subset="validation",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    color_mode='grayscale'
)

# Model definition (simple CNN for TinyML, aiming for < 200k parameters)
model = models.Sequential([
    layers.Conv2D(16, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.summary()

# Check model parameters
num_params = model.count_params()
print(f"Number of model parameters: {num_params}")
if num_params > 200000:
    print("WARNING: Model has more than 200,000 parameters!")

model.compile(optimizer='adam',
			loss='sparse_categorical_crossentropy',
			metrics=['accuracy'])

print("Training model...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
)

# Save the trained model in Keras H5 format
model_h5_path = 'traffic_sign_model.h5'
model.save(model_h5_path)
print(f"Model saved to {model_h5_path}")

# Convert the Keras model to TensorFlow Lite with int8 quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Ensure representative dataset for integer quantization
def representative_dataset():
    # Đảm bảo val_ds đã được định nghĩa ở cell này (hoặc nạp lại data)
    for images, _ in val_ds.unbatch().batch(1).take(100):
        yield [tf.cast(images, tf.float32)]

converter.representative_dataset = representative_dataset

# Ensure that if any ops can't be quantized, the converter throws an error
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8  # or tf.uint8
converter.inference_output_type = tf.int8  # or tf.uint8

tflite_model = converter.convert()

tflite_model_path = 'traffic_sign_model_quantized.tflite'
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_model)
print(f"Quantized TFLite model saved to {tflite_model_path}")

print("Training and TFLite conversion complete.")

Found 9629 files belonging to 10 classes.
Using 6741 files for training.
Found 9629 files belonging to 10 classes.
Using 2888 files for validation.


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 30, 30, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 15, 15, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 13, 13, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 6, 6, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,242 (309.54 KB)

 Trainable params: 79,242 (309.54 KB)

...

Model saved to traffic_sign_model.h5
INFO:tensorflow:Assets written to: C:\Users\ngong\AppData\Local\Temp\tmpj_av5jxe\assets


INFO:tensorflow:Assets written to: C:\Users\ngong\AppData\Local\Temp\tmpj_av5jxe\assets


Saved artifact at 'C:\Users\ngong\AppData\Local\Temp\tmpj_av5jxe'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 1), dtype=tf.float32, name='keras_tensor_24')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  3010256023632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3010256024400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3010256025360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3010256024592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3010256025744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3010256025552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3010256026128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3010256025936: TensorSpec(shape=(), dtype=tf.resource, name=None)


c:\Users\ngong\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Quantized TFLite model saved to traffic_sign_model_quantized.tflite
Training and TFLite conversion complete.


In [7]:
# ==========================================
# TEST TFLITE MODEL & XUẤT CSV (BẢN CHUẨN DUT)
# ==========================================
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf

# 1. Load ĐÚNG file TFLite mới nhất bạn vừa convert thành công
# (Hãy đảm bảo tên file này khớp với file ở Cell Convert)
MODEL_PATH = "traffic_sign_model_quantized.tflite" 
interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

# Lấy thông tin Input/Output
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
input_index = input_details[0]['index']
output_index = output_details[0]['index']

# Lấy thông tin Quantization (Cực kỳ quan trọng cho INT8)
input_scale, input_zero_point = input_details[0]['quantization']

# 2. Quét danh sách ảnh Test
# Lưu ý: Sắp xếp theo tên để đảm bảo thứ tự trong CSV
test_images_paths = glob.glob(os.path.join(TEST_DIR, '*.png'))
test_images_paths.sort() 

results = []

print(f"🚀 Đang xử lý {len(test_images_paths)} ảnh test...")

for img_path in test_images_paths:
    # Lấy ID từ tên file (ví dụ: '001.png' -> '001')
    img_id = os.path.basename(img_path).split('.')[0]
    
    # --- TIỀN XỬ LÝ (Phải khớp 100% với lúc Train) ---
    # Load ảnh dạng GRAYSCALE (vì model của bạn train với color_mode='grayscale')
    img = tf.keras.utils.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE), color_mode='grayscale')
    img_array = tf.keras.utils.img_to_array(img)
    
    # Bước A: Rescaling 1./255 (giống normalization_layer lúc train)
    img_array = img_array / 255.0
    
    # Bước B: Quantization Scaling (Ép float sang INT8)
    if input_scale != 0:
        img_array = (img_array / input_scale) + input_zero_point
    
    # Thêm batch dimension và ép kiểu int8
    input_data = np.expand_dims(img_array.astype(np.int8), axis=0)
    
    # --- CHẠY INFERENCE ---
    interpreter.set_tensor(input_index, input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_index)[0]
    
    # Lấy class có xác suất cao nhất
    predicted_class = np.argmax(output_data)
    
    results.append({'Id': img_id, 'Label': int(predicted_class)})

# 3. XUẤT FILE CSV
submission_df = pd.DataFrame(results)

# Đảm bảo cột Id được lưu đúng định dạng (ví dụ 0001 thay vì 1 nếu cần)
submission_df.to_csv('submission.csv', index=False)

print("-" * 30)
print(f"✅ Đã xuất file thành công: submission.csv")
print(f"🔹 Tổng cộng: {len(submission_df)} dòng.")
print(submission_df.head()) # Xem trước 5 dòng đầu

c:\Users\ngong\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


🚀 Đang xử lý 1545 ảnh test...
------------------------------
✅ Đã xuất file thành công: submission.csv
🔹 Tổng cộng: 1545 dòng.
      Id  Label
0  00000      0
1  00007      0
2  00008      0
3  00012      0
4  00013      0
